In [7]:
"""
Notebook-style Python script to compare teaming results across multiple datasets
(e.g. teaming_uc1_m0.csv ... teaming_uc1_m3.csv).

Save this file and open it as a notebook (Jupyter) or run with `python`.

Outputs:
 - a summary CSV: comparison_summary.csv
 - comparison table printed in the notebook
 - bar plots saved as PNGs

Assumptions about the input CSV format (adapt if needed):
 - columns: researcher_name, team, goodness
 - 'team' column contains a Python-style list (e.g. "['a','b']") or a string of members separated by commas
 - 'goodness' column contains a Python-style list of numeric scores (e.g. "[0.8, 0.6]") or a string with numeric values separated by commas

If your CSV format differs, adapt the parsing functions below.
"""

import ast
import glob
import os
from typing import List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------
# Helper parsing functions
# ----------------------

def parse_list_field(value: object) -> List:
    """Attempt to parse a cell that is meant to be a list.
    Accepts actual list objects, string repr like "['a','b']", or comma-separated strings.
    Returns a Python list (possibly empty).
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, (int, float)):
        return [value]
    s = str(value).strip()
    # try a safe literal eval first (e.g. "['a','b']" or "[0.1, 0.2]")
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
    except Exception:
        pass
    # fallback: split on commas
    if s == '':
        return []
    parts = [p.strip() for p in s.split(',') if p.strip() != '']
    return parts

# ----------------------
# Metrics calculation per dataset
# ----------------------

def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Given a dataframe with columns 'researcher_name', 'team', 'goodness', compute per-researcher and dataset metrics.
    Returns a dictionary-like DataFrame with dataset-level summary and a per-researcher DataFrame.
    """
    # Parse fields into lists
    df = df.copy()
    df['team_list'] = df['team'].apply(parse_list_field)
    df['goodness_list'] = df['goodness'].apply(parse_list_field)

    # Per-row metrics
    df['volume'] = df['team_list'].apply(len)  # #T per row
    df['avg_goodness_per_row'] = df['goodness_list'].apply(lambda lst: float(np.mean(lst)) if len(lst) > 0 else 0.0)

    # Group by researcher
    researcher_stats = df.groupby('researcher_name').agg(
        avg_goodness_per_row=('avg_goodness_per_row', 'mean'),
        volume=('volume', 'mean'),
        row_count=('researcher_name', 'count')
    ).reset_index()

    # Dataset-level metrics
    dataset_summary = {
        'G_mean': researcher_stats['avg_goodness_per_row'].mean(),
        'G_std': researcher_stats['avg_goodness_per_row'].std(ddof=0),
        'Volume_mean': researcher_stats['volume'].mean(),
        'n_researchers': len(researcher_stats),
        'n_rows': len(df)
    }

    return pd.DataFrame([dataset_summary]), researcher_stats

# ----------------------
# Driver: load multiple datasets and compare
# ----------------------

def compare_files(pattern='teaming_uc1_m*.csv', output_dir='compare_outputs'):
    os.makedirs(output_dir, exist_ok=True)
    files = sorted(glob.glob(pattern))
    if len(files) == 0:
        raise FileNotFoundError(f"No files found for pattern: {pattern}")

    summary_rows = []
    per_dataset_researcher_stats = {}

    for path in files:
        name = os.path.basename(path)
        print(f"Processing {name}...")
        df = pd.read_csv(path)
        # if researcher_name column isn't named exactly that, try to infer
        if 'researcher_name' not in df.columns:
            # try common alternates
            candidates = [c for c in df.columns if 'researcher' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'researcher_name'})
            else:
                raise KeyError(f"No 'researcher_name' column found in {path}. Found: {df.columns.tolist()}")

        # same for 'team' and 'goodness'
        if 'team' not in df.columns:
            candidates = [c for c in df.columns if 'team' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'team'})
            else:
                raise KeyError(f"No 'team' column found in {path}. Found: {df.columns.tolist()}")

        if 'goodness' not in df.columns:
            candidates = [c for c in df.columns if 'goodness' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'goodness'})
            else:
                # if not present, introduce a zero goodness
                df['goodness'] = [[] for _ in range(len(df))]

        dataset_summary, researcher_stats = compute_metrics(df)
        dataset_summary['dataset'] = name
        summary_rows.append(dataset_summary.assign(dataset=name))
        per_dataset_researcher_stats[name] = researcher_stats

        # Save researcher-level table
        researcher_stats.to_csv(os.path.join(output_dir, f'{name}_researcher_stats.csv'), index=False)

    # Concatenate summaries
    summary_df = pd.concat(summary_rows, ignore_index=True)
    summary_df = summary_df[['dataset', 'G_mean', 'G_std', 'Volume_mean', 'n_researchers', 'n_rows']]
    summary_df.to_csv(os.path.join(output_dir, 'comparison_summary.csv'), index=False)

    # Print summary
    print('\nComparison summary:')
    print(summary_df.to_string(index=False))

    # Plot comparisons
    plt.figure(figsize=(8, 4))
    plt.bar(summary_df['dataset'], summary_df['G_mean'])
    plt.title('Average Goodness (G) per dataset')
    plt.xlabel('Dataset')
    plt.ylabel('G mean')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'G_mean_comparison.png'))
    plt.close()

    plt.figure(figsize=(8, 4))
    plt.bar(summary_df['dataset'], summary_df['Volume_mean'])
    plt.title('Average Volume (#T) per dataset')
    plt.xlabel('Dataset')
    plt.ylabel('Average Volume')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'Volume_mean_comparison.png'))
    plt.close()

    return summary_df, per_dataset_researcher_stats

# If run as a script, manually specify the CSV files you want to compare
if __name__ == '__main__':

    # >>>>>>> MANUALLY ADD THE CSV FILES YOU WANT TO COMPARE HERE <<<<<<<
    files_to_compare = [
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m0.csv',
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m1.csv',
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m2.csv',
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m3.csv',
        # '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_knapsack.csv',
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m6.csv',
        '../data/v1_output_teaming/teaming_1698proposals_316researchers/teaming_uc1_m7.csv',
        # 'teaming_results_m1.csv',
        # 'teaming_results_m2.csv',
        # 'teaming_results_m3.csv',
        # 'teaming_results_dynamic_knapsack.csv'
        
        
        # Add or remove files as needed
    ]

    # Validate files
    
    for f in files_to_compare:
        if not os.path.exists(f):
            raise FileNotFoundError(f"File not found: {f}")

    summary_rows = []
    per_dataset_researcher_stats = {}

    os.makedirs('compare_outputs', exist_ok=True)

    for path in files_to_compare:
        name = os.path.basename(path)
        print(f"Processing {name}...")
        df = pd.read_csv(path)

        if 'researcher_name' not in df.columns:
            candidates = [c for c in df.columns if 'researcher' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'researcher_name'})
            else:
                raise KeyError(f"No 'researcher_name' column found in {path}. Found: {df.columns.tolist()}")

        if 'team' not in df.columns:
            candidates = [c for c in df.columns if 'team' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'team'})
            else:
                raise KeyError(f"No 'team' column found in {path}. Found: {df.columns.tolist()}")

        if 'goodness' not in df.columns:
            candidates = [c for c in df.columns if 'goodness' in c.lower()]
            if len(candidates) > 0:
                df = df.rename(columns={candidates[0]: 'goodness'})
                
            else:
                df['goodness'] = [[] for _ in range(len(df))]

        dataset_summary, researcher_stats = compute_metrics(df)
        dataset_summary['dataset'] = name
        summary_rows.append(dataset_summary.assign(dataset=name))
        per_dataset_researcher_stats[name] = researcher_stats

        researcher_stats.to_csv(os.path.join('compare_outputs', f'{name}_researcher_stats.csv'), index=False)

    summary_df = pd.concat(summary_rows, ignore_index=True)
    summary_df = summary_df[['dataset', 'G_mean', 'G_std', 'Volume_mean', 'n_researchers', 'n_rows']]
    summary_df.to_csv(os.path.join('compare_outputs', 'comparison_summary.csv'), index=False)

    print('\nComparison summary:')
    print(summary_df.to_string(index=False))

    print('\nSaved outputs to compare_outputs/')

Processing teaming_uc1_m0.csv...
Processing teaming_uc1_m1.csv...
Processing teaming_uc1_m2.csv...
Processing teaming_uc1_m3.csv...
Processing teaming_uc1_m6.csv...
Processing teaming_uc1_m7.csv...

Comparison summary:
           dataset   G_mean    G_std  Volume_mean  n_researchers  n_rows
teaming_uc1_m0.csv 0.087239 0.000478    10.000000            202   87668
teaming_uc1_m1.csv 0.364926 0.055547    10.000000            202   87668
teaming_uc1_m2.csv 0.429701 0.027474     9.999977            202   87668
teaming_uc1_m3.csv 0.589362 0.005798     6.835969            202   87264
teaming_uc1_m6.csv 0.452016 0.007841     5.289146            202   87264
teaming_uc1_m7.csv 0.617562 0.001788     5.559372            202   87264

Saved outputs to compare_outputs/
